# 1.5 前一個字提供什麼資訊？

# 第 1 章：從猜下一個字開始

前置：讀過入門指南的 Python 與 tensor 部分即可。這一章只用幾個字；不要先開完整 Transformer。每段 code 都可單獨執行。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：把分數換成比例：像分一塊餅**

一起加同常數不改比例，分數差才重要。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/softmax.svg")))

**先想一想：**貓之後常見「看」，狗之後常見「喫」，整體頻率能分清嗎？

bigram 就是相鄰的兩個字。每一列記錄「前一個字固定時，下一字各出現幾次」，所以每列要自己除以總數。

**把直覺寫成數學：**$p(j\mid i)=(n_{ij}+1)/(\sum_k n_{ik}+V)$；加 1 避免未見組合機率為零。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
text = "貓看狗。狗喫魚。貓看鳥。"
chars = sorted(set(text))
ids = [chars.index(c) for c in text]
counts = torch.ones(len(chars), len(chars))
for a, b in zip(ids[:-1], ids[1:]):
    counts[a, b] += 1
probability = counts / counts.sum(-1, keepdim=True)
print(chars, probability)
assert torch.allclose(probability.sum(-1), torch.ones(len(chars)))

**如何讀結果：**列是條件，欄是候選答案；加 1 是平滑，不是模型已理解語意。

**只改一件事：**只把平滑常數 1 改成 0.1，比較未見組合。
